# VeReMi-Graph v1.0 — Final Physical Dataset Packager

This notebook creates the public Parquet data layer from the frozen Graph Dataset V1.0.

Source:
`GRAPH_DATASET_ROOT / "archives"`

Release root:
`RELEASE_ROOT`

Output:
- `data/nodes/<archive>.parquet`
- `data/edges/<archive>.parquet`
- `data/snapshots/<archive>.parquet`

It validates the authoritative VeReMi-Graph v1.0 totals:
- 225 archives
- 2,016,487 node rows
- 39,136,291 directed message edges
- 22,500 graph snapshots
- 100 snapshots per archive


In [ ]:
from src.config import (
    GRAPH_DATASET_ROOT,
    BALANCED_SPLIT_ROOT,
    FEATURE_SCHEMA_ROOT,
    RELEASE_ROOT,
)

from pathlib import Path
import sys

# Locate repository root when notebook is executed from notebooks/
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name in {"notebooks", "analysis", "examples"}:
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import hashlib
import shutil
import json

SOURCE_ROOT = GRAPH_DATASET_ROOT / "archives"
RELEASE_ROOT = RELEASE_ROOT

NODE_DIR = RELEASE_ROOT / "data" / "nodes"
EDGE_DIR = RELEASE_ROOT / "data" / "edges"
SNAP_DIR = RELEASE_ROOT / "data" / "snapshots"
META_DIR = RELEASE_ROOT / "metadata"
CHECKSUM_DIR = RELEASE_ROOT / "checksums"

for p in [NODE_DIR, EDGE_DIR, SNAP_DIR, META_DIR, CHECKSUM_DIR]:
    p.mkdir(parents=True, exist_ok=True)

EXPECTED_ARCHIVES = 225
EXPECTED_NODE_ROWS = 2_016_487
EXPECTED_EDGE_ROWS = 39_136_291
EXPECTED_SNAPSHOT_ROWS = 22_500

archive_dirs = sorted(p for p in SOURCE_ROOT.iterdir() if p.is_dir())
print("Archives:", len(archive_dirs))
assert len(archive_dirs) == EXPECTED_ARCHIVES


In [ ]:
# Copy frozen Parquet files into the release layout.
records = []

for i, d in enumerate(archive_dirs, start=1):
    name = d.name
    items = [
        ("node", d/"node_features.parquet", NODE_DIR/f"{name}.parquet"),
        ("edge", d/"edge_features.parquet", EDGE_DIR/f"{name}.parquet"),
        ("snapshot", d/"snapshot_features.parquet", SNAP_DIR/f"{name}.parquet"),
    ]

    for layer, src, dst in items:
        if not src.exists():
            raise FileNotFoundError(src)
        shutil.copy2(src, dst)
        records.append({
            "archive_name": name,
            "layer": layer,
            "release_path": dst.relative_to(RELEASE_ROOT).as_posix(),
            "size_bytes": dst.stat().st_size
        })

    if i == 1 or i % 25 == 0:
        print(f"Copied {i}/{len(archive_dirs)}")

file_manifest = pd.DataFrame(records)
file_manifest.to_csv(META_DIR/"physical_file_manifest.csv", index=False)

print("Release files:", len(file_manifest))
display(file_manifest.head())


In [ ]:
# Validate released row counts and snapshot ranges.
rows = []

for i, d in enumerate(archive_dirs, start=1):
    name = d.name

    n = pd.read_parquet(NODE_DIR/f"{name}.parquet", columns=["snapshot_id","vehicle_id"])
    e = pd.read_parquet(EDGE_DIR/f"{name}.parquet", columns=["snapshot_id"])
    s = pd.read_parquet(SNAP_DIR/f"{name}.parquet", columns=["snapshot_id"])

    rows.append({
        "archive_name": name,
        "node_rows": len(n),
        "edge_rows": len(e),
        "snapshot_rows": len(s),
        "snapshot_count": s["snapshot_id"].nunique(),
        "snapshot_min": s["snapshot_id"].min(),
        "snapshot_max": s["snapshot_id"].max(),
        "unique_vehicles": n["vehicle_id"].nunique()
    })

    if i == 1 or i % 25 == 0:
        print(f"Validated {i}/{len(archive_dirs)}")

validation = pd.DataFrame(rows)
validation.to_csv(META_DIR/"physical_row_validation.csv", index=False)

print("Node rows:", int(validation.node_rows.sum()))
print("Edge rows:", int(validation.edge_rows.sum()))
print("Snapshot rows:", int(validation.snapshot_rows.sum()))

assert int(validation.node_rows.sum()) == EXPECTED_NODE_ROWS
assert int(validation.edge_rows.sum()) == EXPECTED_EDGE_ROWS
assert int(validation.snapshot_rows.sum()) == EXPECTED_SNAPSHOT_ROWS
assert (validation.snapshot_count == 100).all()
assert ((validation.snapshot_min == 0) & (validation.snapshot_max == 99)).all()


In [ ]:
# Public layer summary.
summary = pd.DataFrame({
    "layer": ["nodes","edges","snapshots"],
    "file_count": [
        len(list(NODE_DIR.glob("*.parquet"))),
        len(list(EDGE_DIR.glob("*.parquet"))),
        len(list(SNAP_DIR.glob("*.parquet")))
    ],
    "row_count": [
        int(validation.node_rows.sum()),
        int(validation.edge_rows.sum()),
        int(validation.snapshot_rows.sum())
    ],
    "size_gb": [
        sum(p.stat().st_size for p in NODE_DIR.glob("*.parquet"))/(1024**3),
        sum(p.stat().st_size for p in EDGE_DIR.glob("*.parquet"))/(1024**3),
        sum(p.stat().st_size for p in SNAP_DIR.glob("*.parquet"))/(1024**3)
    ]
})

summary.to_csv(META_DIR/"public_data_layer_summary.csv", index=False)
display(summary)


In [ ]:
# SHA-256 checksums for all 675 data files.
def sha256_file(path, chunk_size=8*1024*1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()

files = sorted(NODE_DIR.glob("*.parquet")) + sorted(EDGE_DIR.glob("*.parquet")) + sorted(SNAP_DIR.glob("*.parquet"))
checksum_rows = []

for i, p in enumerate(files, start=1):
    checksum_rows.append({
        "relative_path": p.relative_to(RELEASE_ROOT).as_posix(),
        "size_bytes": p.stat().st_size,
        "sha256": sha256_file(p)
    })
    if i == 1 or i % 100 == 0:
        print(f"Checksummed {i}/{len(files)}")

checksums = pd.DataFrame(checksum_rows)
checksums.to_csv(CHECKSUM_DIR/"data_sha256.csv", index=False)

assert len(checksums) == 675
print("Checksummed files:", len(checksums))


In [ ]:
# Final validation report.
checks = [
    ("225 node files", len(list(NODE_DIR.glob("*.parquet"))) == 225),
    ("225 edge files", len(list(EDGE_DIR.glob("*.parquet"))) == 225),
    ("225 snapshot files", len(list(SNAP_DIR.glob("*.parquet"))) == 225),
    ("2,016,487 node rows", int(validation.node_rows.sum()) == EXPECTED_NODE_ROWS),
    ("39,136,291 edge rows", int(validation.edge_rows.sum()) == EXPECTED_EDGE_ROWS),
    ("22,500 snapshot rows", int(validation.snapshot_rows.sum()) == EXPECTED_SNAPSHOT_ROWS),
    ("100 snapshots per archive", bool((validation.snapshot_count == 100).all())),
    ("Snapshot IDs 0..99", bool(((validation.snapshot_min == 0) & (validation.snapshot_max == 99)).all())),
    ("675 data checksums", len(checksums) == 675),
]

report = pd.DataFrame(checks, columns=["check","passed"])
report["status"] = np.where(report["passed"], "PASS", "FAIL")
overall = "PASS_PHYSICAL_PACKAGE" if report["passed"].all() else "REVIEW_REQUIRED"

report.loc[len(report)] = ["FINAL_PHYSICAL_PACKAGE_STATUS", report["passed"].all(), overall]
report.to_csv(RELEASE_ROOT/"PHYSICAL_PACKAGE_REPORT.csv", index=False)

display(report)
print("\nFinal status:", overall)
print("Release root:", RELEASE_ROOT)


## Next

When this reports `PASS_PHYSICAL_PACKAGE`, run the companion PyTorch Geometric conversion notebook.
